# BR06 — Final MIMIC-CXR reference-count sensitivity (k=1–5)

Run after BR02 completes for the current cohort. Reuses exactly the same queries, Symile anchors, CXR pools, reverse/graph scores and context cap. Creates separate child runs without editing frozen root Python files. This changes final reference count only. Choose k using validation, then freeze it before test/external performance review. Missing distinct-patient references and context truncation remain explicit. Do not run BR00/BR01 writes inside child banks, which are linked read-only by convention.


In [ ]:
from pathlib import Path
import sys
EXP = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / 'config.example.json').is_file() and (p / 'common.py').is_file()), None)
if EXP is None: raise FileNotFoundError('Start Jupyter inside graph_reasoning/experiment02')
sys.path.insert(0, str(EXP))
from common import load_config, parent_setup, read
CONFIG_PATH = EXP / 'config.json'
if not CONFIG_PATH.is_file():
    raise FileNotFoundError('Run BR00 first to create experiment02/config.json')
CFG = load_config(CONFIG_PATH)
PROJECT = parent_setup(CFG)
RUN = Path(CFG['run_dir'])
print('Kernel:', sys.executable)
print('Cohort:', CFG['cohort'], '| outputs:', RUN)


In [ ]:
from extensions.k_sweep import prepare_sweep
K_VALUES = [1, 2, 3, 4, 5]
SWEEP_VERSION = 'v1'
prepared = prepare_sweep(CFG, K_VALUES, sweep_version=SWEEP_VERSION)
K_CONFIGS = {k: value['config'] for k, value in prepared.items()}
for k, value in prepared.items(): print(k, value['config_path'])


Load one model once, then pilot each k. After all pilots pass, set MAX_NEW_PER_K=0. The total time limit applies across the entire sweep, including all k values. Rerun all cells after a new allocation; successes are skipped. Each model retains its own virtual environment and output budget. For this sensitivity, ECG anchor count stays fixed; only final_cxr changes.


In [ ]:
from generation import model_snapshot, load_generator
MODEL_KEY = 'pulse'  # Start with one generator; expand only after validation supports the additional cost.
MODEL_PATH = None
DOWNLOAD_MODEL = False
ENABLE_THINKING = False
LOAD_IN_4BIT = False
PULSE_REPO = None
MODEL_PATH = model_snapshot(MODEL_KEY, MODEL_PATH, download=DOWNLOAD_MODEL,
                            pin_dir=RUN / 'model_downloads', token_file='/path/to/private/hf_token')
runtime = load_generator(CFG, MODEL_KEY, MODEL_PATH, enable_thinking=ENABLE_THINKING,
                         load_in_4bit=LOAD_IN_4BIT, pulse_repo=PULSE_REPO)


In [ ]:
from extensions.k_sweep import generate_sweep
MAX_NEW_PER_K = 5  # One query across five arms for EACH k; then 0 for all remaining.
RETRY_ERRORS = False
MAX_RUN_HOURS = 30.0  # Total across k values, below remaining allocation time.
MAX_NEW_TOKENS = 1024 if MODEL_KEY == 'qwen38' else 384
print(generate_sweep(K_CONFIGS, runtime, max_new_per_k=MAX_NEW_PER_K,
                     retry_errors=RETRY_ERRORS, max_run_hours=MAX_RUN_HOURS,
                     max_new_tokens=MAX_NEW_TOKENS))


Evaluate in the CPU/graph environment after generation completes. Run the preparation cell to restore K_CONFIGS without loading a model. Choose the same completed model set at every k. ALLOW_PARTIAL=True is an explicit matched-complete-case analysis. The aggregation also computes a common query subset across all k values, so k comparisons need not mix different populations. Larger k can lose evidence to the fixed context cap; inspect context_exposure_by_k.csv.


In [ ]:
from generation import folder_for
from evaluation import evaluate
from queries import load_queries
EVAL_MODELS = ['pulse']  # Expand only to models completed at EVERY k.
ALLOW_PARTIAL = False
MAPPING_REVIEW_NOTE = ''  # Required for reviewed Shaoxing mapping.
EVALUATION_VERSION = 'v1'
REFERENCE_TEXTS = {}
if CFG['cohort'] != 'shaoxing':
    from report_study import reference_texts
    cases = [dict(q, query_id=q['id']) for q in load_queries(CFG)]
    REFERENCE_TEXTS = reference_texts(PROJECT, read(PROJECT / 'config.json'), cases)
FOLDERS_BY_K = {}
DESTINATIONS = {}
for k, child in K_CONFIGS.items():
    folders = {key: folder_for(child, key, False, 1024 if key == 'qwen38' else 384) for key in EVAL_MODELS}
    destination = Path(child['run_dir']) / 'evaluation' / child['cohort'] / EVALUATION_VERSION
    FOLDERS_BY_K[k] = folders
    DESTINATIONS[k] = destination
    result = evaluate(child, folders,
        Path(child['shaoxing_cohort']) / 'targets.json' if child['cohort'] == 'shaoxing' else Path(child['frozen_out']) / 'targets.json',
        destination, reference_texts=REFERENCE_TEXTS, allow_partial=ALLOW_PARTIAL,
        mapping_review_note=MAPPING_REVIEW_NOTE, max_run_hours=5.0)
    print('k:', k, 'complete:', result['complete'])
    if not result['complete']: break


In [ ]:
from extensions.k_sweep import summarize_sweep
SUMMARY = RUN / 'k_sensitivity' / SWEEP_VERSION / CFG['cohort'] / 'summary'
print(summarize_sweep(K_CONFIGS, FOLDERS_BY_K, DESTINATIONS, SUMMARY))
print('Common-query comparison:', SUMMARY / 'common_query_summary_by_k.csv')
print('Actual context exposure:', SUMMARY / 'context_exposure_by_k.csv')
